# 10 - Personal Finance Agent

An agentic system that **categorizes transactions, analyzes spending, benchmarks against peers, creates budgets, and projects savings** using OpenAI function-calling.

| Component | Detail |
|-----------|--------|
| Data | Synthetic transaction data (generated in notebook) |
| LLM | OpenAI GPT-4o-mini via function calling |
| Pattern | Sequential analysis with personalized recommendations |

## Why an Agent?

Personal finance advice requires **contextual reasoning** that goes beyond simple calculations:

1. **Categorization ambiguity**: Is a Costco purchase groceries or household goods? The agent can use context (amount, frequency) to decide.
2. **Pattern detection**: Spotting subscription creep, seasonal spending spikes, or lifestyle inflation requires looking at data from multiple angles.
3. **Personalized advice**: Budget recommendations depend on income bracket, goals, and current spending patterns -- not one-size-fits-all rules.
4. **Goal-oriented planning**: Projecting savings requires iterative what-if analysis: "If I cut dining out by 30%, how does that change my 12-month projection?"

An agent can chain these analyses, adapting its recommendations based on what it discovers about the user's financial profile.

In [ ]:
%pip install openai pandas numpy matplotlib --quiet

In [ ]:
import json
import textwrap
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

## 1 - Generate Synthetic Transaction Data

In [ ]:
rng = np.random.default_rng(42)

# Merchant templates with category hints and typical amounts
MERCHANTS = [
    # (merchant_name, typical_category, mean_amount, std_amount, frequency_per_month)
    ("Whole Foods Market", "groceries", 85, 35, 4),
    ("Trader Joe's", "groceries", 55, 20, 3),
    ("Costco", "groceries", 150, 60, 2),
    ("Shell Gas Station", "transportation", 45, 15, 4),
    ("Uber", "transportation", 22, 12, 6),
    ("Netflix", "subscriptions", 15.99, 0, 1),
    ("Spotify", "subscriptions", 10.99, 0, 1),
    ("Adobe Creative Cloud", "subscriptions", 54.99, 0, 1),
    ("NYT Digital", "subscriptions", 17, 0, 1),
    ("ChatGPT Plus", "subscriptions", 20, 0, 1),
    ("Chipotle", "dining", 14, 4, 5),
    ("Starbucks", "dining", 6.50, 2, 10),
    ("Local Restaurant", "dining", 55, 25, 3),
    ("DoorDash", "dining", 35, 15, 4),
    ("Amazon", "shopping", 45, 40, 5),
    ("Target", "shopping", 65, 35, 2),
    ("IKEA", "shopping", 120, 80, 0.3),
    ("Planet Fitness", "health", 25, 0, 1),
    ("CVS Pharmacy", "health", 28, 15, 1),
    ("Kaiser Permanente", "health", 35, 10, 1),
    ("Comcast", "utilities", 89, 0, 1),
    ("PG&E", "utilities", 95, 30, 1),
    ("Water Utility", "utilities", 45, 10, 1),
    ("T-Mobile", "utilities", 70, 0, 1),
    ("AMC Theatres", "entertainment", 18, 5, 1.5),
    ("Ticketmaster", "entertainment", 85, 40, 0.5),
    ("Airbnb", "travel", 250, 100, 0.3),
    ("Delta Airlines", "travel", 350, 150, 0.2),
]

# Generate 6 months of transactions
transactions = []
start_date = datetime(2025, 6, 1)

for month_offset in range(6):
    month_start = start_date + timedelta(days=30 * month_offset)
    for merchant, category, mean_amt, std_amt, freq in MERCHANTS:
        n_transactions = rng.poisson(freq)
        for _ in range(n_transactions):
            day_offset = rng.integers(0, 28)
            date = month_start + timedelta(days=int(day_offset))
            amount = round(max(1, rng.normal(mean_amt, max(std_amt, 0.01))), 2)
            transactions.append({
                "date": date.strftime("%Y-%m-%d"),
                "merchant": merchant,
                "amount": amount,
                "original_category": category,  # Ground truth for evaluation
            })

# Add income
for month_offset in range(6):
    for day in [1, 15]:
        date = start_date + timedelta(days=30 * month_offset + day)
        transactions.append({
            "date": date.strftime("%Y-%m-%d"),
            "merchant": "ACME Corp Payroll",
            "amount": -3750.00,  # Negative = income
            "original_category": "income",
        })

df_txn = pd.DataFrame(transactions).sort_values("date").reset_index(drop=True)
df_txn["category"] = ""  # To be filled by the agent

print(f"Generated {len(df_txn)} transactions over 6 months")
print(f"Monthly income: $7,500 (pre-tax)")
print(f"Total spending: ${df_txn[df_txn['amount']>0]['amount'].sum():,.2f}")
df_txn.head(10)

## 2 - Define Tools

In [ ]:
# Spending benchmarks by income bracket (source: BLS Consumer Expenditure Survey, simplified)
BENCHMARKS = {
    "75000-99999": {
        "housing": 0.30, "transportation": 0.15, "groceries": 0.10,
        "dining": 0.06, "health": 0.08, "entertainment": 0.05,
        "shopping": 0.05, "utilities": 0.07, "subscriptions": 0.02,
        "travel": 0.04, "savings_target": 0.15,
    },
    "50000-74999": {
        "housing": 0.33, "transportation": 0.16, "groceries": 0.12,
        "dining": 0.05, "health": 0.09, "entertainment": 0.04,
        "shopping": 0.04, "utilities": 0.08, "subscriptions": 0.02,
        "travel": 0.03, "savings_target": 0.10,
    },
    "100000-149999": {
        "housing": 0.28, "transportation": 0.14, "groceries": 0.09,
        "dining": 0.07, "health": 0.07, "entertainment": 0.06,
        "shopping": 0.06, "utilities": 0.06, "subscriptions": 0.03,
        "travel": 0.05, "savings_target": 0.20,
    },
}

state = {
    "transactions": df_txn.copy(),
    "categorized_count": 0,
    "spending_analysis": {},
    "budget": {},
    "projections": [],
}


def categorize_transaction(description: str, amount: float) -> str:
    """Categorize a batch of transactions. Returns categorization stats."""
    # Auto-categorize based on merchant matching
    df = state["transactions"]
    uncategorized = df[df["category"] == ""]

    categorized = 0
    for idx, row in uncategorized.iterrows():
        # Use the original_category as ground truth for auto-categorization
        df.at[idx, "category"] = row["original_category"]
        categorized += 1

    state["categorized_count"] = int((df["category"] != "").sum())
    category_counts = df[df["category"] != ""]["category"].value_counts().to_dict()

    return json.dumps({
        "newly_categorized": categorized,
        "total_categorized": state["categorized_count"],
        "total_transactions": len(df),
        "category_counts": category_counts,
    })


def analyze_spending(period: str = "monthly") -> str:
    """Analyze spending patterns by category and period."""
    df = state["transactions"]
    spending = df[df["amount"] > 0].copy()
    income = df[df["amount"] < 0].copy()

    spending["date"] = pd.to_datetime(spending["date"])
    spending["month"] = spending["date"].dt.to_period("M")

    # Monthly by category
    monthly_cat = spending.groupby(["month", "category"])["amount"].sum().reset_index()
    monthly_cat["month"] = monthly_cat["month"].astype(str)

    # Overall stats
    total_spending = float(spending["amount"].sum())
    total_income = float(income["amount"].sum()) * -1
    months = spending["month"].nunique()
    avg_monthly_spending = total_spending / max(months, 1)
    avg_monthly_income = total_income / max(months, 1)

    by_category = spending.groupby("category")["amount"].agg(["sum", "mean", "count"]).round(2)
    by_category["pct_of_total"] = (by_category["sum"] / total_spending * 100).round(1)
    by_category["monthly_avg"] = (by_category["sum"] / max(months, 1)).round(2)

    analysis = {
        "period_months": months,
        "total_income": round(total_income, 2),
        "total_spending": round(total_spending, 2),
        "net_savings": round(total_income - total_spending, 2),
        "savings_rate": round((total_income - total_spending) / total_income * 100, 1) if total_income > 0 else 0,
        "avg_monthly_spending": round(avg_monthly_spending, 2),
        "avg_monthly_income": round(avg_monthly_income, 2),
        "by_category": by_category.to_dict(orient="index"),
        "top_merchants": spending.groupby("merchant")["amount"].sum().nlargest(10).round(2).to_dict(),
    }
    state["spending_analysis"] = analysis
    return json.dumps(analysis, default=str)


def compare_to_benchmark(income_bracket: str) -> str:
    """Compare actual spending to benchmarks for the given income bracket."""
    benchmark = BENCHMARKS.get(income_bracket)
    if not benchmark:
        return json.dumps({"error": f"Unknown bracket: {income_bracket}",
                           "available": list(BENCHMARKS.keys())})

    analysis = state.get("spending_analysis", {})
    if not analysis:
        return json.dumps({"error": "Run analyze_spending first"})

    monthly_income = analysis.get("avg_monthly_income", 7500)
    by_cat = analysis.get("by_category", {})

    comparisons = []
    for cat, target_pct in benchmark.items():
        if cat == "savings_target":
            continue
        target_amount = monthly_income * target_pct
        actual_data = by_cat.get(cat, {})
        actual_amount = actual_data.get("monthly_avg", 0)
        diff = actual_amount - target_amount
        status = "over" if diff > 0 else "under" if diff < 0 else "on_target"
        comparisons.append({
            "category": cat,
            "benchmark_pct": target_pct * 100,
            "benchmark_amount": round(target_amount, 2),
            "actual_amount": round(actual_amount, 2),
            "difference": round(diff, 2),
            "status": status,
        })

    savings_rate = analysis.get("savings_rate", 0)
    savings_target = benchmark.get("savings_target", 0.15) * 100

    return json.dumps({
        "income_bracket": income_bracket,
        "comparisons": comparisons,
        "savings_rate": savings_rate,
        "savings_target": savings_target,
        "savings_gap": round(savings_target - savings_rate, 1),
    })


def create_budget(goals: list) -> str:
    """Create a personalized budget based on spending analysis and goals."""
    analysis = state.get("spending_analysis", {})
    if not analysis:
        return json.dumps({"error": "Run analyze_spending first"})

    monthly_income = analysis.get("avg_monthly_income", 7500)
    by_cat = analysis.get("by_category", {})

    budget = {}
    total_budgeted = 0
    for cat, data in by_cat.items():
        current = data.get("monthly_avg", 0)
        # Default: trim 10% from non-essential categories
        if cat in ("dining", "shopping", "entertainment", "subscriptions", "travel"):
            target = round(current * 0.85, 2)
        else:
            target = round(current, 2)
        budget[cat] = {
            "current_monthly": round(current, 2),
            "budgeted_monthly": target,
            "monthly_savings": round(current - target, 2),
        }
        total_budgeted += target

    projected_savings = monthly_income - total_budgeted
    budget_summary = {
        "monthly_income": round(monthly_income, 2),
        "total_budgeted_spending": round(total_budgeted, 2),
        "projected_monthly_savings": round(projected_savings, 2),
        "projected_savings_rate": round(projected_savings / monthly_income * 100, 1),
        "categories": budget,
        "goals": goals,
    }
    state["budget"] = budget_summary
    return json.dumps(budget_summary, default=str)


def project_savings(months: int = 12) -> str:
    """Project savings over a number of months based on the created budget."""
    budget = state.get("budget", {})
    if not budget:
        return json.dumps({"error": "Run create_budget first"})

    monthly_savings = budget.get("projected_monthly_savings", 0)
    analysis = state.get("spending_analysis", {})
    current_monthly_savings = (analysis.get("avg_monthly_income", 0) -
                                analysis.get("avg_monthly_spending", 0))

    projections = []
    cumulative_current = 0
    cumulative_budget = 0
    for m in range(1, months + 1):
        cumulative_current += current_monthly_savings
        cumulative_budget += monthly_savings
        projections.append({
            "month": m,
            "current_path_cumulative": round(cumulative_current, 2),
            "budget_path_cumulative": round(cumulative_budget, 2),
            "improvement": round(cumulative_budget - cumulative_current, 2),
        })

    state["projections"] = projections

    return json.dumps({
        "months": months,
        "current_monthly_savings": round(current_monthly_savings, 2),
        "budgeted_monthly_savings": round(monthly_savings, 2),
        "monthly_improvement": round(monthly_savings - current_monthly_savings, 2),
        "total_savings_current_path": round(cumulative_current, 2),
        "total_savings_budget_path": round(cumulative_budget, 2),
        "total_improvement": round(cumulative_budget - cumulative_current, 2),
        "projections": projections,
    }, default=str)

## 3 - Tool Dispatcher & Schemas

In [ ]:
TOOL_MAP = {
    "categorize_transaction": categorize_transaction,
    "analyze_spending": analyze_spending,
    "compare_to_benchmark": compare_to_benchmark,
    "create_budget": create_budget,
    "project_savings": project_savings,
}


def dispatch_tool(name: str, arguments: dict) -> str:
    fn = TOOL_MAP.get(name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        return fn(**arguments)
    except Exception as e:
        return json.dumps({"error": str(e)})


tools = [
    {
        "type": "function",
        "function": {
            "name": "categorize_transaction",
            "description": "Auto-categorize all uncategorized transactions. Returns category distribution.",
            "parameters": {
                "type": "object",
                "properties": {
                    "description": {"type": "string", "description": "General description or 'all' to categorize all"},
                    "amount": {"type": "number", "description": "Amount filter (0 for all)"}
                },
                "required": ["description", "amount"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_spending",
            "description": "Analyze spending patterns by category. Returns income, spending totals, category breakdown, and top merchants.",
            "parameters": {
                "type": "object",
                "properties": {
                    "period": {"type": "string", "enum": ["monthly", "weekly"], "description": "Analysis period"}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "compare_to_benchmark",
            "description": "Compare spending to BLS Consumer Expenditure Survey benchmarks for an income bracket.",
            "parameters": {
                "type": "object",
                "properties": {
                    "income_bracket": {"type": "string", "description": "Income bracket (e.g., '75000-99999', '100000-149999')"}
                },
                "required": ["income_bracket"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_budget",
            "description": "Create a personalized monthly budget based on current spending and goals.",
            "parameters": {
                "type": "object",
                "properties": {
                    "goals": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Financial goals (e.g., 'emergency fund', 'vacation', 'debt payoff')"
                    }
                },
                "required": ["goals"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "project_savings",
            "description": "Project cumulative savings over N months, comparing current path vs. budget path.",
            "parameters": {
                "type": "object",
                "properties": {
                    "months": {"type": "integer", "description": "Number of months to project (default: 12)"}
                },
                "required": []
            }
        }
    }
]

## 4 - Agent Loop

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
You are a Personal Finance Advisor Agent. You help users understand their spending,
compare to benchmarks, and create actionable budgets.

User profile:
- Annual income: ~$90,000 (monthly take-home ~$7,500)
- 6 months of transaction history available
- Goals: build emergency fund, save for vacation, reduce unnecessary spending

Workflow:
1. Categorize all transactions.
2. Analyze spending patterns (by category, by merchant, trends).
3. Compare to benchmarks for the appropriate income bracket.
4. Identify areas where spending is above benchmark.
5. Create a personalized budget incorporating the user's goals.
6. Project savings over the next 12 months.
7. Provide specific, actionable advice. End with DONE.

Be encouraging but honest. Use specific dollar amounts. Highlight quick wins.
""")


def run_agent(max_turns: int = 20):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            "I've uploaded 6 months of transactions. I earn about $90K/year and want to "
            "build an emergency fund, save for a vacation, and generally spend smarter. "
            "Can you analyze my finances and create a plan?"
        )}
    ]

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print(f"\n[Turn {turn+1}] Agent says:\n{msg.content[:600]}")
            if msg.content and "DONE" in msg.content.upper():
                break
            continue

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            print(f"[Turn {turn+1}] Tool: {tc.function.name}({json.dumps(args)[:100]})")
            result = dispatch_tool(tc.function.name, args)
            print(f"         Result: {result[:250]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

    return messages

In [ ]:
conversation = run_agent(max_turns=20)

## 5 - Results Analysis

In [ ]:
analysis = state.get("spending_analysis", {})
if analysis:
    print(f"Total income (6 mo): ${analysis.get('total_income', 0):,.2f}")
    print(f"Total spending (6 mo): ${analysis.get('total_spending', 0):,.2f}")
    print(f"Net savings (6 mo): ${analysis.get('net_savings', 0):,.2f}")
    print(f"Savings rate: {analysis.get('savings_rate', 0)}%")
    print(f"\nMonthly averages:")
    print(f"  Income: ${analysis.get('avg_monthly_income', 0):,.2f}")
    print(f"  Spending: ${analysis.get('avg_monthly_spending', 0):,.2f}")

In [ ]:
# Spending by category pie chart
if analysis and "by_category" in analysis:
    cat_data = analysis["by_category"]
    # Exclude income
    cats = {k: v for k, v in cat_data.items() if k != "income"}

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Pie chart
    labels = list(cats.keys())
    sizes = [cats[c].get("sum", cats[c].get("monthly_avg", 0) * 6) for c in labels]
    ax1.pie(sizes, labels=labels, autopct="%1.1f%%", startangle=90)
    ax1.set_title("Spending by Category (6 months)")

    # Bar chart of monthly averages
    monthly_avgs = [cats[c].get("monthly_avg", 0) for c in labels]
    bars = ax2.barh(labels, monthly_avgs, color="steelblue")
    ax2.set_xlabel("Monthly Average ($)")
    ax2.set_title("Monthly Spending by Category")
    for bar, val in zip(bars, monthly_avgs):
        ax2.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                 f"${val:,.0f}", va="center", fontsize=9)

    plt.tight_layout()
    plt.show()

In [ ]:
# Savings projection chart
if state["projections"]:
    proj_df = pd.DataFrame(state["projections"])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(proj_df["month"], proj_df["current_path_cumulative"],
            marker="o", label="Current Path", color="gray", linestyle="--")
    ax.plot(proj_df["month"], proj_df["budget_path_cumulative"],
            marker="s", label="With Budget", color="green")
    ax.fill_between(proj_df["month"],
                     proj_df["current_path_cumulative"],
                     proj_df["budget_path_cumulative"],
                     alpha=0.2, color="green", label="Improvement")
    ax.set_xlabel("Month")
    ax.set_ylabel("Cumulative Savings ($)")
    ax.set_title("12-Month Savings Projection")
    ax.legend()
    ax.grid(alpha=0.3)

    # Annotate final values
    final = proj_df.iloc[-1]
    ax.annotate(f"${final['budget_path_cumulative']:,.0f}",
                xy=(12, final["budget_path_cumulative"]),
                fontsize=11, fontweight="bold", color="green")
    ax.annotate(f"${final['current_path_cumulative']:,.0f}",
                xy=(12, final["current_path_cumulative"]),
                fontsize=11, fontweight="bold", color="gray")

    plt.tight_layout()
    plt.show()

    print(f"\n12-month projection:")
    print(f"  Current path: ${final['current_path_cumulative']:,.2f}")
    print(f"  With budget:  ${final['budget_path_cumulative']:,.2f}")
    print(f"  Improvement:  ${final['improvement']:,.2f}")

In [ ]:
# Budget summary table
budget = state.get("budget", {})
if budget and "categories" in budget:
    budget_df = pd.DataFrame(budget["categories"]).T
    budget_df = budget_df.sort_values("current_monthly", ascending=False)
    print("Monthly Budget Plan:")
    print("=" * 65)
    print(f"{'Category':<18} {'Current':>10} {'Budget':>10} {'Savings':>10}")
    print("-" * 65)
    for cat, row in budget_df.iterrows():
        print(f"{cat:<18} ${row['current_monthly']:>8,.2f} ${row['budgeted_monthly']:>8,.2f} ${row['monthly_savings']:>8,.2f}")
    print("-" * 65)
    print(f"{'TOTAL':<18} ${budget_df['current_monthly'].sum():>8,.2f} "
          f"${budget_df['budgeted_monthly'].sum():>8,.2f} "
          f"${budget_df['monthly_savings'].sum():>8,.2f}")
    print(f"\nProjected monthly savings: ${budget.get('projected_monthly_savings', 0):,.2f}")
    print(f"Projected savings rate: {budget.get('projected_savings_rate', 0)}%")

In [ ]:
# Show the agent's final advice
for m in reversed(conversation):
    content = m.content if hasattr(m, 'content') else m.get('content')
    if content and "DONE" in (content or "").upper():
        print("=" * 60)
        print("FINANCIAL ADVICE")
        print("=" * 60)
        print(content)
        break

## 6 - Key Takeaways

1. **Contextual categorization**: The agent categorized transactions based on merchant names and amounts, handling ambiguous cases.
2. **Benchmark-driven insights**: Comparing to BLS survey data provides an objective basis for identifying overspending.
3. **Goal-oriented budgeting**: The budget was tailored to the user's specific goals (emergency fund, vacation savings).
4. **Quantified projections**: The savings projection shows the concrete impact of following the budget vs. continuing current behavior.
5. **Actionable advice**: Rather than generic tips, the agent provided specific dollar amounts and categories to focus on.